In [17]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

In [2]:
file = "data/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2024.csv"

df = pd.read_csv(file)

C:\Users\alnex\AppData\Local\Temp\ipykernel_12680\2828549565.py:3: DtypeWarning: Columns (0: Rndrng_Prvdr_State_FIPS, 1: Rndrng_Prvdr_Zip5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


In [4]:
df['services_per_bene'] = (df['Tot_Srvcs']/df['Tot_Benes'].replace(0, np.nan))
df['services_per_bene_day'] = (df['Tot_Srvcs']/df['Tot_Bene_Day_Srvcs'].replace(0, np.nan))
df['charge_to_allowed'] = (df['Avg_Sbmtd_Chrg']/df['Avg_Mdcr_Alowd_Amt'].replace(0, np.nan))
df["payment_to_allowed"] = (df["Avg_Mdcr_Pymt_Amt"] /df["Avg_Mdcr_Alowd_Amt"].replace(0, np.nan))
df.replace([np.inf, -np.inf], np.nan, inplace=True)

,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,...,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt,services_per_bene,services_per_bene_day,charge_to_allowed,payment_to_allowed
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,36.0,36,284.378889,79.063889,60.828889,62.376389,1.000000,1.0,3.596824,0.769364
1,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,150.0,150,352.176333,126.340800,95.675600,96.007133,1.000000,1.0,2.787511,0.757282
2,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,63.0,63,783.997302,170.360000,134.300476,131.999365,1.000000,1.0,4.602003,0.788333
3,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,16.0,16,122.125000,47.832500,33.350625,33.706250,1.454545,1.0,2.553180,0.697238
4,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,70.0,70,583.606857,81.428571,62.728714,58.486857,1.891892,1.0,7.167102,0.770353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9781668,1992999825,Deschenes,Geoffrey,R,M.D.,I,1100 9th Ave,Ms:m4-Pfs,Seattle,WA,...,92.0,92,330.000000,140.840326,104.658370,93.407065,1.150000,1.0,2.343079,0.743099
9781669,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,36.0,36,780.583333,165.405000,131.781389,133.834444,1.000000,1.0,4.719225,0.796719
9781670,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,329.0,329,278.127660,74.811337,59.662888,61.002462,1.750000,1.0,3.717721,0.797511
9781671,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,153.0,153,398.522876,113.171438,89.568366,90.994967,1.545455,1.0,3.521409,0.791440


In [13]:
model_features = [
    'Tot_Benes',
    'services_per_bene',
    'services_per_bene_day',
    'charge_to_allowed',
    'payment_to_allowed',
    'Avg_Sbmtd_Chrg',
]

In [6]:
peer_cols = ['Rndrng_Prvdr_Type', 'HCPCS_Cd', 'Place_Of_Srvc']

In [8]:
df['peer_group_size'] = (df.groupby(peer_cols)['Rndrng_NPI'].transform('size'))
df['peer_group_size'].describe()

count    9.781673e+06
mean     1.166123e+04
std      1.761181e+04
min      1.000000e+00
25%      1.153000e+03
50%      5.176000e+03
75%      1.296200e+04
max      9.133000e+04
Name: peer_group_size, dtype: float64

In [10]:
for size in [20, 50, 100, 200]:
    pct = (df['peer_group_size'] >= size).mean() * 100
    print(f'Peer group with at least {size} rows : {pct:.2f}% of dataset')

Peer group with at least 20 rows : 98.23% of dataset
Peer group with at least 50 rows : 96.63% of dataset
Peer group with at least 100 rows : 94.56% of dataset
Peer group with at least 200 rows : 91.41% of dataset


In [14]:
df['iforest_raw_score'] = np.nan
df['iforest_risk_score'] = np.nan
df['iforest_modeled'] = False
min_peer_size = 100

In [21]:
grouped = df.groupby(peer_cols).groups
group_counter = 0
for peer_name, peer_index in grouped.items():
    group_counter += 1
    group = df.loc[peer_index]
    valid = group.dropna(subset=model_features)

    if len(valid) < min_peer_size:
        continue

    X = valid[model_features].copy()
    X = np.log1p(X)

    model = IsolationForest(
        n_estimators=100,
        max_samples='auto',
        random_state=42,
        n_jobs=1,
    )

    model.fit(X)

    raw_scores = -model.score_samples(X)
    df.loc[valid.index, 'iforest_raw_score'] = raw_scores

    risk_scores = (
        pd.Series(raw_scores, index=valid.index).rank(pct=True)*100
    )

    df.loc[valid.index, 'iforest_risk_score'] = risk_scores
    df.loc[valid.index, 'iforest_modeled'] = True
    if group_counter % 500 == 0:
        print(f'Processed {group_counter} peer groups')

Processed 2500 peer groups
Processed 4500 peer groups
Processed 5000 peer groups
Processed 11000 peer groups
Processed 25500 peer groups
Processed 26500 peer groups
Processed 36000 peer groups
Processed 38500 peer groups
Processed 41000 peer groups
Processed 41500 peer groups
Processed 46500 peer groups
Processed 47000 peer groups
Processed 51000 peer groups
Processed 51500 peer groups


In [22]:
print("Total rows:", len(df))
print("Modeled rows:", df["iforest_modeled"].sum())

print(
    "Percent modeled:",
    round(df["iforest_modeled"].mean() * 100, 2),
    "%"
)

Total rows: 9781673
Modeled rows: 9249744
Percent modeled: 94.56 %


In [24]:
example_peer = df[
    (df["Rndrng_Prvdr_Type"] == "Family Practice") &
    (df["HCPCS_Cd"] == "99214") &
    (df["Place_Of_Srvc"] == "O") &
    (df["iforest_modeled"] == True)
].copy()

In [27]:
top10 = example_peer[
    [
        "Rndrng_NPI",
        "Tot_Benes",
        "Tot_Srvcs",
        "services_per_bene",
        "services_per_bene_day",
        "charge_to_allowed",
        "payment_to_allowed",
        "Avg_Mdcr_Stdzd_Amt",
        "iforest_raw_score",
        "iforest_risk_score"
    ]
].sort_values(
    "iforest_risk_score",
    ascending=False
).head(10)

top10

,Rndrng_NPI,Tot_Benes,Tot_Srvcs,services_per_bene,services_per_bene_day,charge_to_allowed,payment_to_allowed,Avg_Mdcr_Stdzd_Amt,iforest_raw_score,iforest_risk_score
655798,1063709962,39,43.0,1.102564,1.023810,11.268489,0.405534,50.932093,0.733265,100.000000
6783664,1699168328,76,152.0,2.000000,1.041096,11.715997,0.628156,78.475000,0.713131,99.998304
4670880,1477713139,20,36.0,1.800000,1.028571,11.745137,0.642114,80.566389,0.709762,99.996608
3632248,1376040709,63,107.0,1.698413,1.019048,11.710597,0.569660,71.425607,0.701892,99.994912
7625570,1770899346,78,138.0,1.769231,1.007299,11.691935,0.553273,69.484493,0.700362,99.993216
6686432,1689170730,44,55.0,1.250000,1.018519,11.697842,0.557261,69.990364,0.700178,99.991520
3962974,1407350556,14,23.0,1.642857,1.045455,5.177893,0.471387,72.552174,0.699373,99.989824
8505050,1861885196,54,76.0,1.407407,1.027027,11.754701,0.614312,76.809342,0.698407,99.988129
6575072,1679065387,11,11.0,1.000000,1.000000,1.227006,0.075963,11.290909,0.692840,99.986433
2062247,1215049051,24,24.0,1.000000,1.000000,5.358531,0.122329,18.288333,0.691969,99.984737


In [28]:
peer_medians = example_peer[
    [
        "Tot_Benes",
        "Tot_Srvcs",
        "services_per_bene",
        "services_per_bene_day",
        "charge_to_allowed",
        "payment_to_allowed",
        "Avg_Mdcr_Stdzd_Amt"
    ]
].median()

peer_medians

Tot_Benes                 97.000000
Tot_Srvcs                167.000000
services_per_bene          1.675862
services_per_bene_day      1.000000
charge_to_allowed          2.098019
payment_to_allowed         0.660485
Avg_Mdcr_Stdzd_Amt        83.756805
dtype: float64

In [29]:
highest_risk = (
    example_peer
    .sort_values("iforest_risk_score", ascending=False)
    .iloc[0]
)

print("Risk score:", highest_risk["iforest_risk_score"])

print(
    "Services per beneficiary:",
    highest_risk["services_per_bene"],
    "| Peer median:",
    peer_medians["services_per_bene"]
)

print(
    "Charge ratio:",
    highest_risk["charge_to_allowed"],
    "| Peer median:",
    peer_medians["charge_to_allowed"]
)

print(
    "Beneficiaries:",
    highest_risk["Tot_Benes"],
    "| Peer median:",
    peer_medians["Tot_Benes"]
)

Risk score: 100.0
Services per beneficiary: 1.1025641025641026 | Peer median: 1.6758620689655173
Charge ratio: 11.268488594497512 | Peer median: 2.0980191766591147
Beneficiaries: 39 | Peer median: 97.0
